# Gemini Vision Basics

**HFA AI Training — Module 6: Vision Language Models**

Google Gemini has excellent vision capabilities, especially for:
- Video analysis and understanding (best in class)
- Processing many images at once (huge context window)
- Spatial reasoning (where things are in an image)
- Structured data extraction from documents and photos

This notebook demonstrates:
1. Sending an image to Gemini from a URL
2. Sending a local image to Gemini
3. Analyzing a photo and getting a description
4. Extracting structured data from a document image

**Get a free API key at:** https://aistudio.google.com/apikey

## Install Dependencies

In [ ]:
!pip install google-generativeai httpx pillow

## Imports and Configuration

Load the required libraries and configure the Google Gemini API key from environment variables.
Never hard-code API keys — always use environment variables.

In [ ]:
import os
import io
import json
import httpx
from pathlib import Path

import google.generativeai as genai
from PIL import Image

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Always load API keys from environment variables — never hard-code them.
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY not set. Run: export GOOGLE_API_KEY='your-key'\n"
        "Get a free key at https://aistudio.google.com/apikey"
    )

# Configure the SDK with our API key.
genai.configure(api_key=GOOGLE_API_KEY)

# Use the latest Gemini model with vision support.
# Gemini 2.0 Flash is fast, capable, and cost-effective for vision tasks.
MODEL_NAME = "gemini-2.0-flash"

## Helper: Load Image from URL

Gemini can accept images as raw bytes or PIL Image objects.
This helper downloads an image from a URL and returns it as bytes.

In [ ]:
def load_image_from_url(image_url: str) -> bytes:
    """
    Download an image from a URL and return the raw bytes.

    Gemini accepts images as raw bytes (with a mime_type) or as PIL
    Image objects. This helper handles downloading from a URL.

    Args:
        image_url: Public URL of the image.

    Returns:
        The raw image bytes.
    """
    response = httpx.get(image_url)
    response.raise_for_status()
    return response.content

## Example 1: Analyze an Image from a URL

Download an image from a URL and ask Gemini to analyze it.
This is the simplest way to get started — just point Gemini at an image
and ask a question about it.

In [ ]:
def analyze_image_from_url(image_url: str, question: str) -> str:
    """
    Download an image from a URL and ask Gemini to analyze it.

    This is the simplest way to get started — just point Gemini at an image
    and ask a question about it.

    Args:
        image_url: Public URL of the image.
        question: What you want to know about the image.

    Returns:
        Gemini's analysis text.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 1: Analyze an image from URL")
    print(f"{'='*60}")
    print(f"Image URL: {image_url}")
    print(f"Question:  {question}")

    # Download the image bytes from the URL.
    image_bytes = load_image_from_url(image_url)

    # Create the Gemini model instance.
    model = genai.GenerativeModel(MODEL_NAME)

    # Send the image and question to Gemini.
    # Gemini accepts images as raw bytes with a mime_type dict.
    response = model.generate_content(
        [
            {"mime_type": "image/jpeg", "data": image_bytes},
            question,
        ]
    )

    result = response.text
    print(f"\nGemini's Analysis:\n{result}")
    return result

## Example 2: Analyze a Local Image File

Load an image from a local file using PIL and send it to Gemini.
Gemini's SDK accepts PIL Image objects directly — no base64 encoding needed.

In [ ]:
def analyze_local_image(image_path: str, question: str) -> str:
    """
    Load an image from a local file and ask Gemini about it.

    Uses PIL (Pillow) to open the image, which Gemini's SDK accepts directly.
    No base64 encoding needed — just open the file and send it.

    Args:
        image_path: Path to a local image file (jpg, png, etc.).
        question: What you want to know about the image.

    Returns:
        Gemini's analysis text.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 2: Analyze a local image file")
    print(f"{'='*60}")
    print(f"Image path: {image_path}")
    print(f"Question:   {question}")

    # Open the image with PIL. Gemini's SDK can accept PIL Image objects directly.
    image = Image.open(image_path)

    model = genai.GenerativeModel(MODEL_NAME)

    # Pass the PIL Image directly — the SDK handles the conversion.
    response = model.generate_content([image, question])

    result = response.text
    print(f"\nGemini's Analysis:\n{result}")
    return result

## Example 3: Extract Structured Data from a Document Image

Gemini excels at document parsing — this is "OCR on steroids."
Instead of just reading text, Gemini UNDERSTANDS the document structure
and can organize the information into clean JSON.

Great for:
- Receipts -> line items, totals, tax
- Business cards -> name, phone, email, company
- Invoices -> vendor, items, amounts, due date
- Property listings -> features, price, dimensions

In [ ]:
def extract_structured_data(image_url: str) -> dict:
    """
    Ask Gemini to extract structured JSON data from a document image.

    Gemini excels at document parsing — this is "OCR on steroids."
    Instead of just reading text, Gemini UNDERSTANDS the document structure
    and can organize the information into clean JSON.

    Great for:
      - Receipts -> line items, totals, tax
      - Business cards -> name, phone, email, company
      - Invoices -> vendor, items, amounts, due date
      - Property listings -> features, price, dimensions

    Args:
        image_url: URL of a document image.

    Returns:
        Parsed dictionary of extracted data.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 3: Structured data extraction from a document")
    print(f"{'='*60}")

    # Download the image bytes from the URL.
    image_bytes = load_image_from_url(image_url)

    model = genai.GenerativeModel(MODEL_NAME)

    # The prompt is key — be specific about the JSON structure you want.
    extraction_prompt = """Analyze this image carefully and extract all
    information into a structured JSON format.

    If this is a property or real estate image:
    {
        "property_type": "house/condo/apartment/land",
        "style": "architectural style description",
        "condition_assessment": "detailed condition notes",
        "features": ["list", "of", "visible", "features"],
        "materials": ["visible construction materials"],
        "potential_issues": ["any concerns visible"],
        "listing_highlights": ["top selling points for a listing"],
        "estimated_era": "approximate decade or period of construction"
    }

    If this is a document (receipt, form, invoice, etc.):
    {
        "document_type": "type of document",
        "key_fields": { ... extracted fields ... },
        "line_items": [ ... if applicable ... ],
        "totals": { ... if applicable ... }
    }

    Return ONLY valid JSON — no markdown, no extra text."""

    response = model.generate_content(
        [
            {"mime_type": "image/jpeg", "data": image_bytes},
            extraction_prompt,
        ]
    )

    raw_text = response.text.strip()

    # Gemini sometimes wraps JSON in markdown code blocks.
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]
        raw_text = raw_text.rsplit("```", 1)[0]

    try:
        structured_data = json.loads(raw_text)
        print(f"\nExtracted structured data:")
        print(json.dumps(structured_data, indent=2))
        return structured_data
    except json.JSONDecodeError:
        print(f"\nGemini returned text (not strict JSON):\n{raw_text}")
        return {"raw_response": raw_text}

## Example 4: Multi-Image Comparison

Send two images to Gemini and ask it to compare them.

Useful for:
- Before/after comparisons
- Comparing two properties side by side
- Spotting differences between document versions

In [ ]:
def compare_two_images(image_url_1: str, image_url_2: str) -> str:
    """
    Send two images to Gemini and ask it to compare them.

    Useful for:
      - Before/after comparisons
      - Comparing two properties side by side
      - Spotting differences between document versions

    Args:
        image_url_1: URL of the first image.
        image_url_2: URL of the second image.

    Returns:
        Gemini's comparison analysis.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 4: Compare two images")
    print(f"{'='*60}")

    # Download both images.
    image_bytes_1 = load_image_from_url(image_url_1)
    image_bytes_2 = load_image_from_url(image_url_2)

    model = genai.GenerativeModel(MODEL_NAME)

    # Send both images along with a comparison prompt.
    response = model.generate_content(
        [
            {"mime_type": "image/jpeg", "data": image_bytes_1},
            {"mime_type": "image/jpeg", "data": image_bytes_2},
            (
                "Compare these two images in detail. "
                "What are the similarities and differences? "
                "If these are properties, which would you "
                "rate higher and why?"
            ),
        ]
    )

    result = response.text
    print(f"\nGemini's Comparison:\n{result}")
    return result

## Run All Examples

Execute all the Gemini vision examples using sample public domain house photos.

In [ ]:
print("=" * 60)
print("  GEMINI VISION BASICS")
print("  Google Gemini — Fast, Capable Vision Analysis")
print("=" * 60)

# A sample image URL — a public domain house photo.
sample_image_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "1/19/Weatherboard_house_in_Coorparoo%2C_Queensland.jpg/"
    "1280px-Weatherboard_house_in_Coorparoo%2C_Queensland.jpg"
)

### Example 1: Analyze image from URL

In [ ]:
analyze_image_from_url(
    image_url=sample_image_url,
    question="Describe this property in detail, as if you were writing a real estate listing.",
)

### Example 2: Analyze local image

Uncomment and provide a real file path to test.

In [ ]:
# analyze_local_image(
#     image_path="/path/to/your/image.jpg",
#     question="What do you see in this image?",
# )

### Example 3: Structured data extraction

In [ ]:
extract_structured_data(sample_image_url)

### Example 4: Compare two images

In [ ]:
second_image_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "e/e1/Ascot_house_1.jpg/"
    "1280px-Ascot_house_1.jpg"
)
compare_two_images(sample_image_url, second_image_url)

In [ ]:
print("\n" + "=" * 60)
print("  All examples complete!")
print("=" * 60)